In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [ ]:
df = pd.read_csv("powerplant_data.csv")

In [ ]:
df.head(10)

In [ ]:
df.isnull().sum()

In [ ]:
x = df.drop("PE" , axis =1)

In [ ]:
y = df["PE"]

In [ ]:
y.head()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
        x,y, test_size = .2, random_state=42
    )

In [ ]:
df.shape

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Prepareing TensorData

In [ ]:
x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [ ]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Deep Learning

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            # Hidden Layer 1
            nn.Linear(x_train.shape[1],6),
            nn.ReLU(),
            # Hidden Layer 2
            nn.Linear(6,6),
            nn.ReLU(),
            #Output Layer
            nn.Linear(6,1),
        )
    def forward(self, x):
        return self.model(x)
    

In [ ]:
model = ANN()

In [ ]:
# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

***For every batch:***

    Input Batch (feature_batch)
            ↓
    Forward Pass
            ↓
    Predictions
            ↓
    Loss Calculation
            ↓
    Backpropagation
            ↓
    Gradients
            ↓
    Adam Optimizer
            ↓
    Updated Weights

***For every epoch:***

    Train on all batches
            ↓
    Compute average train loss
            ↓
    Evaluate on validation set
            ↓
    Compute average validation loss
            ↓
    Save best model if improved

In [ ]:
# Lists to store the training and validation losses for each epoch
train_loss = []
val_loss = []

best_val_loss = float("inf")

epochs = 100

for epoch in range (epochs):
    model.train()
    running_loss = 0.0

    for feature_batch, label_batch in train_loader:
        optimizer.zero_grad() # Remove the old value

        outputs = model(feature_batch)
        loss = criterion(outputs, label_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    epoch_train_loss = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for feature_batch, label_batch in test_loader:
            outputs = model(feature_batch)
            loss = criterion(outputs, label_batch)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_loss.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")
    
    

In [ ]:
loss_df = pd.DataFrame({
    "Training Loss": train_loss,
    "Validation Loss": val_loss
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
# Loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
print(model)

## Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    train_preds = model(x_train_tensor)
    test_preds = model(x_test_tensor)
    
    train_mse_loss = crietrion(train_preds, y_train_tensor)
    test_mse_loss = crietrion(test_preds, y_test_tensor)  

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

In [ ]:
print("r^2 score =", r2_score(y_test, test_preds))

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)